# Fairness Postprocessing Tutorial

This tutorial demonstrates how to apply the **fairness postprocessing** method in PipeGenie.

Unlike preprocessing, postprocessing does **not modify the training data**. Instead, it mitigates unfairness **after** the model has been trained by applying Fairlearn's **GridSearch** algorithm to each pipeline in the PipeGenie ensemble.

The best mitigated estimator is then selected based on its predictive performance.

## 1. Load and Prepare the Dataset

We begin by:

- Loading the Employee Promotion dataset.
- Defining the target variable (`is_promoted`).
- Selecting the sensitive feature (`education`).
- Splitting the dataset into training and testing sets.

The sensitive feature is **not** used for model training but is required for fairness evaluation and mitigation.

In [1]:
import pandas as pd

df = pd.read_csv("../datasets/employee_promotion.csv")

y = df["is_promoted"]

df["education"] = df["education"].fillna("Unknown")
A = df["education"]

X = df.drop(columns=["is_promoted","education"])
X = pd.get_dummies(X, drop_first=True)

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## 2. Train the PipeGenie Classifier

Next, we create and train a standard PipeGenie classifier using the classification grammar.

At this stage, **no fairness mitigation has been applied**.

In [1]:
grammar_file = "../../grammars/sample-grammar-classification.xml"

from pipegenie.classification import PipegenieClassifier
model = PipegenieClassifier(
        grammar=grammar_file,
        generations=5,
        pop_size=50,
        elite_size=5,
        seed=42,
        outdir="samples/classification_post")

In [4]:
model.fit(X_train,y_train)

                                       fitness                                            size                                        fitness_elite                                      size_elite              
                 ----------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max           avg           std           min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.50973       0.71735       0.65785       0.076887      1      5      2.64          1.3381        0.50973       0.71735       0.65785       0.076887      1      4      2             1.2649    
1      50        0.50973       0.71735       0.6657        0.07235       1      5      2.74          1.3388        0.66107       0.71735       0.69689       0.0

## 3. Apply Fairness Postprocessing

After the ensemble has been trained, fairness mitigation is applied to each pipeline using **Fairlearn's GridSearch**.

Each mitigated estimator is stored for later evaluation.

In [5]:
from pipegenie.fairness import fairness_postprocessing

fair_estimators = fairness_postprocessing(ensemble=model.ensemble, X_train=X_train, y_train=y_train, sensitive_features=A_train)

## 4. Select the Best Mitigated Estimator

Several mitigated estimators are produced during the postprocessing step.

The helper function `select_best_mitigator()` evaluates each estimator on a validation set and returns the one with the best predictive performance.

In [6]:
from pipegenie.fairness import select_best_mitigator

best_name, best_mitigator, best_rmse = select_best_mitigator(fair_estimators, X_test, y_test)

## 5. Generate Predictions

Predictions are generated using the selected mitigated estimator.

These predictions are then used to evaluate both model performance and fairness.

In [7]:
y_pred = best_mitigator.predict(X_test)

## 6. Generate the Fairness Report

The fairness report summarizes:

- Overall model performance
- Performance for each sensitive group

This provides insight into whether the mitigation strategy improved fairness across the selected sensitive feature.

In [8]:
from pipegenie.fairness import fairness_report
from pipegenie.metrics import balanced_accuracy_score

report = fairness_report( y_true=y_test, y_pred=y_pred, sensitive_features=A_test, metric=balanced_accuracy_score)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])


OVERALL PERFORMANCE
0.548688922409602

PERFORMANCE BY GROUP
sensitive_feature_0
Bachelors          0.548327
Below Secondary    0.580443
Mastersabove       0.546403
Unknown            0.568951
Name: balanced_accuracy_score, dtype: float64


## 7. Compute Fairness Metrics

Finally, we compute commonly used fairness metrics:

- **Balanced Accuracy**
- **Equalized Odds Difference**
- **Demographic Parity Difference**

These metrics allow us to compare predictive performance and fairness after postprocessing.

In [9]:
# =====================================================
# STANDARD METRICS
# =====================================================

from fairlearn.metrics import (equalized_odds_difference,demographic_parity_difference)


print("\n==============================")
print("BALANCED ACCURACY")
print("==============================")

print( balanced_accuracy_score(y_test, y_pred))

eo = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

dp = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

print("\n===== FAIRNESS METRICS =====")
print("Equalized Odds Difference:", eo)
print("Demographic Parity Difference:", dp)


BALANCED ACCURACY
0.548688922409602

===== FAIRNESS METRICS =====
Equalized Odds Difference: 0.06072351421188629
Demographic Parity Difference: 0.005185664699095399


## Conclusion

In this tutorial, we:

- Trained a standard PipeGenie classifier.
- Applied fairness postprocessing using Fairlearn's GridSearch algorithm.
- Selected the best mitigated estimator.
- Generated predictions using the selected estimator.
- Evaluated both predictive performance and fairness.

This workflow demonstrates how fairness can be improved **after** model training without modifying the original training data.